In [34]:
import importlib
from io import StringIO
from pathlib import Path
from IPython.core.magic import register_cell_magic
from magic_codec.grammar import parse_grammar_str, generate_python
from pegen.tokenizer import Tokenizer
from pegen.parser import memoize, memoize_left_rec, logger, Parser

from magic_codec.grammar.parser_generator import ParserGenerator, PatchParserGenerator
from magic_codec.parser.peg import *
from tokenize import TokenInfo, generate_tokens

grammar: dict[str, Grammar] = {}
parser: dict[str, any] = {}

@register_cell_magic
def peg(name, cell):
    parsed = parse_grammar_str(cell)
    print(parsed)
    global grammar
    grammar[name] = parsed

@register_cell_magic
def parse(args, cell):
    name, *rules = args.split(' ')

    # print(list((a.string, a.type) for a in generate_tokens(StringIO(cell).readline)))
    # return
    class_name = grammar[name].metas['class']
    _parser = globals()[class_name](Tokenizer(generate_tokens(StringIO(cell).readline)), verbose=True)
    result = None
    if rules:
        assert len(rules) == 1, "Too many args"
        result = getattr(_parser, rules[0])()
    else:
        result = _parser.start()
    print(result)

def generate(name: str, print_code=False):
    global parser
    code = ""
    if 'extends' in grammar[name].metas:
        parent_file = Path(grammar[name].metas['extends'])
        assert parent_file.exists(), "Base grammar does not exist"
        spec = importlib.util.spec_from_file_location(parent_file.stem, parent_file)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        base = grammar[name].metas["base"]
        code = PatchParserGenerator(grammar[name], getattr(module, base)).generate("")
    else:
        code = ParserGenerator(grammar[name]).generate("")
    
    parser[name] = code
    if print_code:
        print(parser[name])
    exec(parser[name], globals=globals())


In [35]:
class Replacement:
    if sys.version_info >= (3, 10):
        __match_args__ = ("name")

    name: str
    _fields = ("name",)
    _field_types = {'name': str}

    def __init__(self, name):
        self.name = name.string
        self.string = f"${name.string}$"
    
    def __repr__(self):
        return f"{self.name}"

    __str__ = __repr__

class Fragment:
    if sys.version_info >= (3, 10):
        __match_args__ = ("data")

    data: list[TokenInfo | Replacement]
    _fields = ("data",)
    _field_types = {'data': list[TokenInfo | Replacement]}

    def __init__(self, *args):
        self.data = list(self.flatten(args))

    def __repr__(self):
        return str([e.string for e in self.data])

    __str__ = __repr__

    def flatten(self, nested_list):
        for item in nested_list:
            if item is None:
                continue
            if isinstance(item, list):
                yield from self.flatten(item)
            elif isinstance(item, Fragment):
                yield from self.flatten(item.data)
            else:
                yield item

def unparse_last_token(parser):
    parser._tokenizer._index = max(0, parser._tokenizer._index - 1)

In [36]:
%%peg test
# Minimal patch grammar for declarative macros
@extends "../../src/magic_codec/parser/peg.py"
@class MacroPegParser
@base PegParser
@trailer''

ANY: ~ { t if (t:=self._tokenizer.getnext()).type != 0 else None }
braces: '(' | ')' | '[' | ']' | '{' | '}'

unparsed_atom:
    | '$' a=NAME { Replacement(a) }
    | a=(!braces !NEWLINE !'$' ANY) { Fragment(a) }

unparsed_balanced:
    | a='(' b=unparsed_balanced? c=')' d=unparsed_balanced? { Fragment(a, b, c, d) } 
    | a='[' b=unparsed_balanced? c=']' d=unparsed_balanced? { Fragment(a, b, c, d) }
    | a='{' b=unparsed_balanced? c='}' d=unparsed_balanced? { Fragment(a, b, c, d) }
    | a=INDENT NEWLINE b=unparsed_balanced c=DEDENT { Fragment(a, b, c) }
    | a=unparsed_atom b=unparsed_balanced? { Fragment(a, b) }

unparsed_block:
    | a=INDENT ~ b=unparsed_block+ c=DEDENT { Fragment(a, b, c) }
    | !INDENT a=unparsed_balanced b=NEWLINE { Fragment(a, b) }

suite_fragment:
    | NEWLINE INDENT ~ a=unparsed_block+ &DEDENT { Fragment(a) }
    | !INDENT !DEDENT a=unparsed_balanced &(a=NEWLINE) { Fragment(a) }

action[list]: 
    | ':' ~ a=suite_fragment { (unparse_last_token(self),a)[1] }
    | !':' "{" ~ a=unparsed_balanced "}" { Fragment(a) }


ANY: ~
braces: '(' | ')' | '[' | ']' | '{' | '}'
unparsed_atom: '$' NAME | (!braces !NEWLINE !'$' ANY)
unparsed_balanced:
    | '(' unparsed_balanced? ')' unparsed_balanced?
    | '[' unparsed_balanced? ']' unparsed_balanced?
    | '{' unparsed_balanced? '}' unparsed_balanced?
    | INDENT NEWLINE unparsed_balanced DEDENT
    | unparsed_atom unparsed_balanced?
unparsed_block: INDENT ~ unparsed_block+ DEDENT | !INDENT unparsed_balanced NEWLINE
suite_fragment:
    | NEWLINE INDENT ~ unparsed_block+ &DEDENT
    | !INDENT !DEDENT unparsed_balanced &(NEWLINE)
action: ':' ~ suite_fragment | !':' "{" ~ unparsed_balanced "}"


In [37]:
generate("test", True)

#!/usr/bin/env python3.8
# @generated by pegen from 

import ast
import sys
import tokenize

from typing import Any, Optional

from pegen.parser import memoize, memoize_left_rec, logger, Parser
class MacroPegParser(PegParser, ):

    @memoize
    def ANY(self) -> Optional[Any]:
        # ANY: ~
        mark = self._mark()
        cut = False
        if (
            (cut := True)
        ):
            return t if ( t := self . _tokenizer . getnext ( ) ) . type != 0 else None;
        self._reset(mark)
        if cut:
            return None;
        return None;

    @memoize
    def braces(self) -> Optional[Any]:
        # braces: '(' | ')' | '[' | ']' | '{' | '}'
        mark = self._mark()
        if (
            (literal := self.expect('('))
        ):
            return literal;
        self._reset(mark)
        if (
            (literal := self.expect(')'))
        ):
            return literal;
        self._reset(mark)
        if (
            (literal := self.expect('['))
  

In [38]:
%%parse test action
: print($a)

action() ... (looking at 1.0: OP:':')
  expect(':') ... (looking at 1.0: OP:':')
  ... expect(':') -> TokenInfo(type=55 (OP), string=':', start=(1, 0), end=(1, 1), line=': print($a)\n')
  suite_fragment() ... (looking at 1.2: NAME:'print')
    expect('NEWLINE') ... (looking at 1.2: NAME:'print')
    ... expect('NEWLINE') -> None
    expect('INDENT') ... (looking at 1.2: NAME:'print')
    ... expect('INDENT') -> None
    expect('DEDENT') ... (looking at 1.2: NAME:'print')
    ... expect('DEDENT') -> None
    unparsed_balanced() ... (looking at 1.2: NAME:'print')
      expect('(') ... (looking at 1.2: NAME:'print')
      ... expect('(') -> None
      expect('[') ... (looking at 1.2: NAME:'print')
      ... expect('[') -> None
      expect('{') ... (looking at 1.2: NAME:'print')
      ... expect('{') -> None
      expect('INDENT') -> None
      unparsed_atom() ... (looking at 1.2: NAME:'print')
        expect('$') ... (looking at 1.2: NAME:'print')
        ... expect('$') -> None
        

In [39]:
%%parse test action
: 
  if True:
    print("x")
  else:
    print("y")
  print("z")

action() ... (looking at 1.0: OP:':')
  expect(':') ... (looking at 1.0: OP:':')
  ... expect(':') -> TokenInfo(type=55 (OP), string=':', start=(1, 0), end=(1, 1), line=': \n')
  suite_fragment() ... (looking at 1.2: NEWLINE:'\n')
    expect('NEWLINE') ... (looking at 1.2: NEWLINE:'\n')
    ... expect('NEWLINE') -> TokenInfo(type=4 (NEWLINE), string='\n', start=(1, 2), end=(1, 3), line=': \n')
    expect('INDENT') ... (looking at 2.0: INDENT:'  ')
    ... expect('INDENT') -> TokenInfo(type=5 (INDENT), string='  ', start=(2, 0), end=(2, 2), line='  if True:\n')
    _loop1_1003() ... (looking at 2.2: NAME:'if')
      unparsed_block() ... (looking at 2.2: NAME:'if')
        expect('INDENT') ... (looking at 2.2: NAME:'if')
        ... expect('INDENT') -> None
        expect('INDENT') -> None
        unparsed_balanced() ... (looking at 2.2: NAME:'if')
          expect('(') ... (looking at 2.2: NAME:'if')
          ... expect('(') -> None
          expect('[') ... (looking at 2.2: NAME:'if')

In [40]:
%%parse test alt
a=NAME: 
  if True:
    print("x")
  else:
    print("y")
  print("z")

alt() ... (looking at 1.0: NAME:'a')
  items() ... (looking at 1.0: NAME:'a')
    named_item() ... (looking at 1.0: NAME:'a')
      name() ... (looking at 1.0: NAME:'a')
      ... name() -> TokenInfo(type=1 (NAME), string='a', start=(1, 0), end=(1, 1), line='a=NAME: \n')
      annotation() ... (looking at 1.1: OP:'=')
        expect('[') ... (looking at 1.1: OP:'=')
        ... expect('[') -> None
      ... annotation() -> None
      name() -> TokenInfo(type=1 (NAME), string='a', start=(1, 0), end=(1, 1), line='a=NAME: \n')
      expect('=') ... (looking at 1.1: OP:'=')
      ... expect('=') -> TokenInfo(type=55 (OP), string='=', start=(1, 1), end=(1, 2), line='a=NAME: \n')
      item() ... (looking at 1.2: NAME:'NAME')
        expect('[') ... (looking at 1.2: NAME:'NAME')
        ... expect('[') -> None
        atom() ... (looking at 1.2: NAME:'NAME')
          expect('(') ... (looking at 1.2: NAME:'NAME')
          ... expect('(') -> None
          name() ... (looking at 1.2: NAME:'N

In [45]:
%%parse test more_alts
| a=NAME:
  block
| a=STRING: 
  b

more_alts() ... (looking at 1.0: OP:'|')
  expect('|') ... (looking at 1.0: OP:'|')
  ... expect('|') -> TokenInfo(type=55 (OP), string='|', start=(1, 0), end=(1, 1), line='| a=NAME:\n')
  alts() ... (looking at 1.2: NAME:'a')
    alt() ... (looking at 1.2: NAME:'a')
      items() ... (looking at 1.2: NAME:'a')
        named_item() ... (looking at 1.2: NAME:'a')
          name() ... (looking at 1.2: NAME:'a')
          ... name() -> TokenInfo(type=1 (NAME), string='a', start=(1, 2), end=(1, 3), line='| a=NAME:\n')
          annotation() ... (looking at 1.3: OP:'=')
            expect('[') ... (looking at 1.3: OP:'=')
            ... expect('[') -> None
          ... annotation() -> None
          name() -> TokenInfo(type=1 (NAME), string='a', start=(1, 2), end=(1, 3), line='| a=NAME:\n')
          expect('=') ... (looking at 1.3: OP:'=')
          ... expect('=') -> TokenInfo(type=55 (OP), string='=', start=(1, 3), end=(1, 4), line='| a=NAME:\n')
          item() ... (looking at 1.4: N